# Phase 4.5-lite Axis-Limit Investigation

Derives cohort-pooled `xlim`, `ylim`, and `x_range` per-(bodypart, center_mode)
for Phase 5/6 heatmaps. Pools the pre-computed 2D histograms from the
alignment-sweep caches at `notebooks/experiments/outputs/alignment_sweep/`
(8 animals × ~21 sessions each × all bodyparts × 4 regimes).

**Regime mapping:** sweep `eye_centered` = production `eye_midpoint`;
sweep `spout_centered` = production `spout_midpoint`. The sweep stores
histograms in canonical-reference space (raw + (ref_midpoint - session_midpoint));
production centering subtracts the per-session midpoint. So
`production_centered = sweep_canonical - reference_midpoint_per_animal`.

**Strategy:** 99.98% quantile (`q00001_9999`) + 40% symmetric padding,
rounded to nearest 5 px, clamped to any legacy x-filter range.

**Legacy x-filters (verified from legacy notebooks, 2026-04-20):**
- `jaw`:    `x_range = [-80, 80]`   (from `20251117_DLC_analysis_jaw.ipynb`)
- `tongue`: `x_range = [-175, 175]` (from `20260323_DLC_build-parquet_reward-aligned_tongue_jaw.ipynb`)

**Outputs** go to `notebooks/experiments/outputs/phase_4.5_lite_axis_limits/`:
per-bodypart pooled heatmap + candidate-overlay + side-by-side cropping
comparison panels, plus `candidate_limits_table.csv`.

See `docs/MIGRATION_LOG.md` Phase 4.5-lite entry for the final approved
values and the Priya-approved decision trail.

In [1]:
# ============================================================
# USER INPUTS
# ============================================================

# Recommended strategy (Priya-approved 2026-04-20):
QUANTILE = (0.0001, 0.9999)   # q00001_9999: 0.01% to 99.99%
PAD_FRAC = 0.40                # 40% symmetric padding on each axis

# Legacy x-filters applied before percentile (clamps to the filter range).
X_FILTER_CENTERED = {
    "jaw":    (-80, 80),       # eye_midpoint frame
    "tongue": (-175, 175),     # spout_midpoint frame
}

BODYPARTS_EYE = ["jaw", "nose", "L_whiskers_1", "L_whiskers_2", "L_whiskers_3",
                 "R_whiskers_1", "R_whiskers_2", "R_whiskers_3"]
TONGUE_BP = "tongue"
ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

In [2]:
import pickle
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CACHE_DIR = Path("outputs/alignment_sweep/_animal_cache")
OUT_DIR = Path("outputs/phase_4.5_lite_axis_limits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Common regrid: centered-space target grid.
COMMON_X = np.arange(-700, 701, 5)   # 280 bins @ 5 px
COMMON_Y = np.arange(-700, 701, 5)


def regrid_to_common(xedges, yedges, hist, ref_xy, target_x, target_y, x_filter=None):
    """Shift sweep histogram to centered frame and regrid to common axis."""
    xe_c = xedges - ref_xy[0]
    ye_c = yedges - ref_xy[1]
    xc = 0.5 * (xe_c[:-1] + xe_c[1:])
    yc = 0.5 * (ye_c[:-1] + ye_c[1:])
    ix = np.searchsorted(target_x, xc) - 1
    iy = np.searchsorted(target_y, yc) - 1
    out = np.zeros((len(target_x) - 1, len(target_y) - 1), dtype=np.float64)
    for i, sx in enumerate(ix):
        if sx < 0 or sx >= out.shape[0]:
            continue
        if x_filter is not None and not (x_filter[0] <= xc[i] <= x_filter[1]):
            continue
        for j, sy in enumerate(iy):
            if sy < 0 or sy >= out.shape[1]:
                continue
            out[sx, sy] += hist[i, j]
    return out


def compute_limits_quantile(pooled_hist, xedges, yedges, lo_q, hi_q):
    """Cumulative-mass quantile bounds of pooled-histogram marginals."""
    total = pooled_hist.sum()
    if total == 0:
        return (None, None), (None, None)
    sx = pooled_hist.sum(axis=1) / total
    sy = pooled_hist.sum(axis=0) / total
    cx, cy = np.cumsum(sx), np.cumsum(sy)
    xc = 0.5 * (xedges[:-1] + xedges[1:])
    yc = 0.5 * (yedges[:-1] + yedges[1:])
    return ((float(xc[np.searchsorted(cx, lo_q)]),
             float(xc[np.searchsorted(cx, min(hi_q, 1 - 1e-9))])),
            (float(yc[np.searchsorted(cy, lo_q)]),
             float(yc[np.searchsorted(cy, min(hi_q, 1 - 1e-9))])))


def padded_clamped_limits(pooled_hist, xedges, yedges, quantile, pad_frac, x_clamp=None):
    (xlo, xhi), (ylo, yhi) = compute_limits_quantile(pooled_hist, xedges, yedges, *quantile)
    xs, ys = xhi - xlo, yhi - ylo
    pxl, pxh = xlo - pad_frac * xs, xhi + pad_frac * xs
    pyl, pyh = ylo - pad_frac * ys, yhi + pad_frac * ys
    if x_clamp is not None:
        pxl = max(pxl, x_clamp[0])
        pxh = min(pxh, x_clamp[1])
    r = lambda v: 5 * round(v / 5)
    return (r(pxl), r(pxh)), (r(pyl), r(pyh))

In [3]:
# Pool eye_midpoint histograms across cohort for jaw/nose/whiskers.
pooled_eye = {bp: np.zeros((len(COMMON_X) - 1, len(COMMON_Y) - 1)) for bp in BODYPARTS_EYE}
n_per_bp = {bp: 0 for bp in BODYPARTS_EYE}

for animal in ANIMALS:
    pkl = CACHE_DIR / f"{animal}.pkl"
    if not pkl.exists():
        continue
    with open(pkl, "rb") as f:
        d = pickle.load(f)
    ref_xy = d["reference"]["eye_mid_ref"]
    for bp in BODYPARTS_EYE:
        entry = d["histograms"].get(bp, {}).get("eye_centered")
        if entry is None:
            continue
        by_date = entry["by_date"]
        ah = np.zeros((len(entry["xedges"]) - 1, len(entry["yedges"]) - 1))
        for h in by_date.values():
            if h is not None:
                ah += h
        if ah.sum() == 0:
            continue
        pooled_eye[bp] += regrid_to_common(
            entry["xedges"], entry["yedges"], ah, ref_xy,
            COMMON_X, COMMON_Y, x_filter=X_FILTER_CENTERED.get(bp),
        )
        n_per_bp[bp] += 1

# Pool tongue under spout_centered.
pooled_tongue = np.zeros((len(COMMON_X) - 1, len(COMMON_Y) - 1))
tongue_n = 0
for animal in ANIMALS:
    pkl = CACHE_DIR / f"{animal}.pkl"
    if not pkl.exists():
        continue
    with open(pkl, "rb") as f:
        d = pickle.load(f)
    ref_spout = d["reference"].get("spout_mid_ref")
    entry = d["histograms"].get(TONGUE_BP, {}).get("spout_centered")
    if ref_spout is None or entry is None:
        continue
    by_date = entry["by_date"]
    ah = np.zeros((len(entry["xedges"]) - 1, len(entry["yedges"]) - 1))
    for h in by_date.values():
        if h is not None:
            ah += h
    if ah.sum() == 0:
        continue
    pooled_tongue += regrid_to_common(
        entry["xedges"], entry["yedges"], ah, ref_spout,
        COMMON_X, COMMON_Y, x_filter=X_FILTER_CENTERED.get(TONGUE_BP),
    )
    tongue_n += 1

print(f"pooled eye_midpoint bodyparts: {n_per_bp}")
print(f"pooled tongue (spout_midpoint): {tongue_n}/{len(ANIMALS)} animals")

pooled eye_midpoint bodyparts: {'jaw': 8, 'nose': 8, 'L_whiskers_1': 8, 'L_whiskers_2': 8, 'L_whiskers_3': 8, 'R_whiskers_1': 8, 'R_whiskers_2': 8, 'R_whiskers_3': 8}
pooled tongue (spout_midpoint): 8/8 animals


In [4]:
# Compute final proposed limits per bodypart under the approved strategy.
final_rows = []
for bp in BODYPARTS_EYE:
    h = pooled_eye[bp]
    if h.sum() == 0:
        continue
    xl, yl = padded_clamped_limits(
        h, COMMON_X, COMMON_Y, QUANTILE, PAD_FRAC,
        x_clamp=X_FILTER_CENTERED.get(bp),
    )
    final_rows.append({
        "bodypart": bp, "center_mode": "eye_midpoint",
        "xlim": [xl[0], xl[1]], "ylim": [yl[0], yl[1]],
        "x_range": list(X_FILTER_CENTERED[bp]) if bp in X_FILTER_CENTERED else None,
        "n_animals": n_per_bp[bp],
    })
if tongue_n > 0:
    xl, yl = padded_clamped_limits(
        pooled_tongue, COMMON_X, COMMON_Y, QUANTILE, PAD_FRAC,
        x_clamp=X_FILTER_CENTERED.get(TONGUE_BP),
    )
    final_rows.append({
        "bodypart": TONGUE_BP, "center_mode": "spout_midpoint",
        "xlim": [xl[0], xl[1]], "ylim": [yl[0], yl[1]],
        "x_range": list(X_FILTER_CENTERED[TONGUE_BP]),
        "n_animals": tongue_n,
    })

final_df = pd.DataFrame(final_rows)
final_df.to_csv(OUT_DIR / "proposed_limits_final.csv", index=False)
print(f"\nFinal proposed limits (q00001_9999 + 40% pad, rounded to 5, x-clamped):\n")
for _, r in final_df.iterrows():
    xlim_s = f"[{r.xlim[0]:.0f}, {r.xlim[1]:.0f}]"
    ylim_s = f"[{r.ylim[0]:.0f}, {r.ylim[1]:.0f}]"
    xr_s = f"x_range={r.x_range}" if r.x_range else "(no x_range)"
    print(f"  {r.bodypart:16s} ({r.center_mode:15s}): xlim={xlim_s} ylim={ylim_s} {xr_s}")


Final proposed limits (q00001_9999 + 40% pad, rounded to 5, x-clamped):

  jaw              (eye_midpoint   ): xlim=[-80, 80] ylim=[195, 520] x_range=[-80, 80]
  nose             (eye_midpoint   ): xlim=[-100, 95] ylim=[100, 330] (no x_range)
  L_whiskers_1     (eye_midpoint   ): xlim=[15, 135] ylim=[55, 250] (no x_range)
  L_whiskers_2     (eye_midpoint   ): xlim=[15, 165] ylim=[145, 355] (no x_range)
  L_whiskers_3     (eye_midpoint   ): xlim=[60, 165] ylim=[75, 270] (no x_range)
  R_whiskers_1     (eye_midpoint   ): xlim=[-140, -15] ylim=[70, 235] (no x_range)
  R_whiskers_2     (eye_midpoint   ): xlim=[-165, 0] ylim=[155, 350] (no x_range)
  R_whiskers_3     (eye_midpoint   ): xlim=[-185, -40] ylim=[95, 265] (no x_range)
  tongue           (spout_midpoint ): xlim=[-175, 175] ylim=[-235, 225] x_range=[-175, 175]


In [5]:
# Render per-bodypart pooled heatmap with recommended-limit rectangle.
# y-axis inverted per camera-view convention (higher y = lower on plot).
def render_one(bp, pooled, frame, limits_row):
    norm_h = pooled / max(pooled.sum(), 1)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(norm_h.T, origin="lower",
              extent=(COMMON_X[0], COMMON_X[-1], COMMON_Y[0], COMMON_Y[-1]),
              aspect="auto", cmap="magma",
              norm=mcolors.LogNorm(vmin=max(norm_h[norm_h > 0].min(), 1e-7), vmax=norm_h.max()))
    xl, yl = limits_row.xlim, limits_row.ylim
    rect = mpatches.Rectangle((xl[0], yl[0]), xl[1] - xl[0], yl[1] - yl[0],
                              fill=False, edgecolor="cyan", linewidth=2,
                              label=f"APPROVED: x={xl} y={yl}")
    ax.add_patch(rect)
    ax.set_title(f"{bp} ({frame}): pooled {limits_row.n_animals} animals × all dates")
    ax.set_xlabel(f"{bp}_x (centered, {frame})")
    ax.set_ylabel(f"{bp}_y (centered, {frame})")
    ax.legend(loc="lower right", fontsize=8)
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"approved_{bp}_{frame}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

for _, r in final_df.iterrows():
    if r.center_mode == "spout_midpoint":
        render_one(r.bodypart, pooled_tongue, r.center_mode, r)
    else:
        render_one(r.bodypart, pooled_eye[r.bodypart], r.center_mode, r)

print(f"Rendered {len(final_df)} per-bodypart approved-limit overlays to {OUT_DIR}")

Rendered 9 per-bodypart approved-limit overlays to outputs\phase_4.5_lite_axis_limits
